# QLoRA training (Colab)

Run this on a Colab GPU runtime (Runtime > Change runtime type > GPU).

This notebook runs the **real 4-bit QLoRA** training path. It reuses the exact same
code as the repo (`train/qlora_train.py`) so the Colab run and the local CPU smoke
test can never drift: when CUDA is present, `qlora_train.py` loads the model in 4-bit
with Unsloth and trains with TRL's `SFTTrainer`; on CPU it falls back to plain `peft`
LoRA (that fallback is what proved the plumbing on Day 2).

Prereqs (Day 2, done): Behavior Spec written, data pipeline in `data/`, eval harness
in `eval/`, and the end-to-end smoke test passing. Day 3 uses this for the first real
training run on the v1 dataset.

## 1. Install environment

In [ ]:
!pip install -q unsloth trl datasets textstat  # textstat: needed by data.audit_v4r3 (readability gate) before training

## 2. Load base model and confirm GPU inference works

In [ ]:
from unsloth import FastLanguageModel

MODEL_ID = "Qwen/Qwen3-4B"  # tune target upgraded 0.6B -> 4B to clear the accuracy floor
                            # (0.6B was capacity-bound: litmus accuracy 5/12). 4B is the CAP.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=2048,
    load_in_4bit=True,   # QLoRA 4-bit: ~2.2GB weights; full run peaks ~6-8GB, fits a free T4 (16GB)
)
FastLanguageModel.for_inference(model)

In [ ]:
messages = [{"role": "user", "content": "In one sentence, what is a QLoRA fine-tune?"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## 3. Train on the generated dataset (real QLoRA)

Clone the repo so the notebook runs the *same* code and dataset as everything else,
then invoke the training module. On a GPU runtime this takes the Unsloth 4-bit QLoRA
path automatically.

In [ ]:
# Get the repo so the notebook runs the *same* code + dataset as everything else.
%cd /content
![ -d SmallLearningModel ] || git clone https://github.com/Elitelord/SmallLearningModel.git
%cd SmallLearningModel
!git pull  # make sure you have the latest committed dataset + code

# QLoRA run on the v4r4 gold dataset (Qwen3-4B). Unsloth 4-bit + TRL SFTTrainer on GPU.
# v4r4 = v4r3 readability target + A3 mechanism-preserving generation, scaled to 600 ex.
# LoRA capacity raised r16/a32 -> r32/a64 (B4) to fit readability AND accuracy together.
# --data must be COMMITTED & PUSHED (Colab clones from GitHub; local-only files are invisible).
# v4r3 kept for comparison: data/v4/gold_v4_r3.jsonl / --adapter-name v4r3 / r16 a32.
!python -m data.audit_v4r3 data/v4/gold_v4_r4.jsonl
!python -m train.qlora_train \
    --data data/v4/gold_v4_r4.jsonl \
    --adapter-name v4r4 \
    --epochs 3 \
    --lora-r 32 --lora-alpha 64 \
    --batch-size 8 --grad-accum 2 --lr 2e-4

## 4. Base-vs-tuned eval

Score the base model and the freshly trained adapter on the held-out concepts with the
identical minimal prompt (`Explain: {concept}`), reusing the litmus **v4** readability
gate (whole-passage FK 3–6 AND ARI 3–7) + the 0/1/2 accuracy judge. Both sides are
served with `enable_thinking=False`, matching training.

The cell below installs the eval deps and runs `--no-judge` first (FK/ARI deltas, no
API). To add the accuracy delta, set the gateway creds (via a Colab Secret) and pass
`--judge openai-group/gpt-4.1` as shown in the commented lines.

In [ ]:
# Eval needs the readability scorer (textstat); the accuracy judge also needs openai.
!pip install -q textstat openai

# Readability-only runs: original 12 litmus concepts and the full 24 held-out concepts.
# Adapter name MUST match the training cell (v4r4).
!python -m eval.base_vs_tuned --adapter train/adapters/v4r4 --eval-key eval_litmus --limit 0 --no-judge --out eval/base_vs_tuned_v4r4_litmus12_readability.json
!python -m eval.base_vs_tuned --adapter train/adapters/v4r4 --eval-key eval --limit 0 --no-judge --out eval/base_vs_tuned_v4r4_all24_readability.json

# ACCURACY: fastest path is to score the saved generations LOCALLY afterward — the
# results JSON stores base_text/tuned_text, so judge them from your own machine with
# `python -m eval.judge_saved <file> --judge openai-group/gpt-4.1` (no Colab gateway setup).
# Or, to judge here, set gateway creds (Colab Secret — do NOT hardcode; this notebook
# is pushed to GitHub) and pass the slug:
#   import os
#   from google.colab import userdata
#   os.environ["OPENAI_API_KEY"]  = userdata.get("TFY_KEY")
#   os.environ["OPENAI_BASE_URL"] = "https://tfy.promptlens.trilogy.com"
#   !python -m eval.base_vs_tuned --adapter train/adapters/v4r4 --eval-key eval_litmus --limit 0 --judge openai-group/gpt-4.1 --out eval/base_vs_tuned_v4r4_litmus12_judged.json
#   !python -m eval.base_vs_tuned --adapter train/adapters/v4r4 --eval-key eval --limit 0 --judge openai-group/gpt-4.1 --out eval/base_vs_tuned_v4r4_all24_judged.json